# VSP-3D Reconstruction — Pipeline Demo

Synthetic DICOM loading → preprocessing → segmentation → mesh extraction → STL export.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('imports ok')

In [ ]:
# --- load synthetic DICOM volume ---

def create_synthetic_ct_volume(shape=(64, 128, 128), seed=42):
    """
    Mimics a CT scan: background noise + a bright ellipsoidal 'bone' region.
    Returns Hounsfield-unit-like values in [-1000, 3000].
    """
    rng = np.random.default_rng(seed)
    volume = rng.normal(-500, 80, shape).astype(np.float32)   # soft tissue background

    # insert synthetic bone ellipsoid
    z, y, x = np.indices(shape)
    z0, y0, x0 = shape[0]//2, shape[1]//2, shape[2]//2
    mask = ((z - z0)**2 / 12**2 + (y - y0)**2 / 30**2 + (x - x0)**2 / 25**2) <= 1.0
    volume[mask] = rng.normal(800, 120, mask.sum())   # cortical bone HU range

    return volume, mask


volume, ground_truth_mask = create_synthetic_ct_volume(shape=(64, 128, 128))

print(f'Volume shape : {volume.shape}')
print(f'HU range     : [{volume.min():.0f}, {volume.max():.0f}]')
print(f'Voxel spacing: (0.5, 0.5, 0.5) mm  [synthetic]')
print(f'Bone voxels  : {ground_truth_mask.sum():,}')

In [ ]:
# --- preprocessing ---

def window_and_normalise(volume: np.ndarray, wl: float = 400, ww: float = 1500) -> np.ndarray:
    """Standard CT bone windowing, then scale to [0, 1]."""
    lo, hi = wl - ww / 2, wl + ww / 2
    clipped = np.clip(volume, lo, hi)
    return (clipped - lo) / (hi - lo)


def remove_table_artifacts(volume: np.ndarray, threshold: float = 0.05) -> np.ndarray:
    """Zero out extreme-low-HU voxels (scanner table, air pockets)."""
    out = volume.copy()
    out[volume < threshold] = 0.0
    return out


preprocessed = window_and_normalise(volume)
preprocessed = remove_table_artifacts(preprocessed)

print('Preprocessing steps applied:')
print(f'  1. bone window  (WL=400, WW=1500)')
print(f'  2. normalise    → [0, 1]')
print(f'  3. table artifact removal')
print(f'Output range    : [{preprocessed.min():.3f}, {preprocessed.max():.3f}]')
print(f'Non-zero voxels : {(preprocessed > 0).sum():,}')

In [ ]:
# --- segmentation inference (sliding-window dummy) ---

def sliding_window_inference(
    volume: np.ndarray,
    patch_size: tuple = (32, 64, 64),
    threshold: float = 0.65,
) -> np.ndarray:
    """
    Simulates nnU-Net-style sliding-window inference.
    Returns a binary segmentation mask.
    """
    prob_map = np.zeros_like(volume, dtype=np.float32)

    # mock probability: high where intensity is in bone range
    bone_prob = np.clip((volume - 0.55) / 0.35, 0, 1)
    # add slight spatial smoothing via box filter
    from scipy.ndimage import uniform_filter
    prob_map = uniform_filter(bone_prob, size=5)

    mask = prob_map >= threshold
    return mask, prob_map


from scipy.ndimage import uniform_filter

seg_mask, prob_map = sliding_window_inference(preprocessed, threshold=0.60)

# compute dice with ground truth
intersection = (seg_mask & ground_truth_mask).sum()
dice = 2 * intersection / (seg_mask.sum() + ground_truth_mask.sum() + 1e-6)

print(f'Segmentation results')
print(f'  Predicted voxels : {seg_mask.sum():,}')
print(f'  Ground-truth vox : {ground_truth_mask.sum():,}')
print(f'  Dice coefficient : {dice:.4f}')
print(f'  Mean probability : {prob_map[ground_truth_mask].mean():.4f}')

In [ ]:
# --- mesh extraction with marching cubes ---

try:
    from skimage.measure import marching_cubes
    verts, faces, normals, values = marching_cubes(seg_mask.astype(np.float32), level=0.5, spacing=(0.5, 0.5, 0.5))
    print('Mesh extracted with skimage.measure.marching_cubes')
except ImportError:
    # fallback: trivial dummy mesh
    print('skimage not available — using dummy mesh')
    verts = np.array([[0,0,0],[1,0,0],[0,1,0],[0,0,1]], dtype=np.float32)
    faces = np.array([[0,1,2],[0,1,3],[0,2,3],[1,2,3]])
    normals = np.ones((4,3), dtype=np.float32) / np.sqrt(3)

print(f'\nMesh statistics:')
print(f'  Vertices : {len(verts):,}')
print(f'  Faces    : {len(faces):,}')
if len(verts) > 0:
    bbox_mm = verts.max(axis=0) - verts.min(axis=0)
    print(f'  Bounding box (mm): {bbox_mm[0]:.1f} x {bbox_mm[1]:.1f} x {bbox_mm[2]:.1f}')

In [ ]:
# --- mesh quality checks ---

def mesh_quality_report(verts, faces):
    report = {}
    report['n_verts'] = len(verts)
    report['n_faces'] = len(faces)

    # edge-length distribution
    if len(faces) > 0:
        v0 = verts[faces[:, 0]]
        v1 = verts[faces[:, 1]]
        v2 = verts[faces[:, 2]]
        e01 = np.linalg.norm(v1 - v0, axis=1)
        e12 = np.linalg.norm(v2 - v1, axis=1)
        e20 = np.linalg.norm(v0 - v2, axis=1)
        all_edges = np.concatenate([e01, e12, e20])
        report['edge_mean_mm'] = float(all_edges.mean())
        report['edge_max_mm']  = float(all_edges.max())
        report['degenerate_faces'] = int((all_edges < 1e-6).sum())
    else:
        report['edge_mean_mm'] = 0.0
        report['edge_max_mm'] = 0.0
        report['degenerate_faces'] = 0

    report['passes_qc'] = report['degenerate_faces'] == 0 and report['n_faces'] > 10
    return report


qc = mesh_quality_report(verts, faces)
print('Mesh QC Report')
print('=' * 40)
for k, v in qc.items():
    print(f'  {k:<25}: {v}')

In [ ]:
# --- export STL ---

import struct
import os

def write_stl_binary(filepath: str, verts: np.ndarray, faces: np.ndarray, normals: np.ndarray):
    """Write a minimal binary STL file."""
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'wb') as f:
        f.write(b'\x00' * 80)               # header
        f.write(struct.pack('<I', len(faces)))  # triangle count
        for i, tri in enumerate(faces):
            n = normals[i] if i < len(normals) else np.array([0, 0, 1], dtype=np.float32)
            f.write(struct.pack('<3f', *n))
            for vi in tri:
                f.write(struct.pack('<3f', *verts[vi]))
            f.write(struct.pack('<H', 0))    # attribute byte count


out_path = '/tmp/vsp_demo_bone.stl'
write_stl_binary(out_path, verts, faces, normals)

size_kb = os.path.getsize(out_path) / 1024
print(f'STL exported to: {out_path}')
print(f'File size      : {size_kb:.1f} KB')
print(f'Triangles      : {len(faces):,}')
print(f'QC passed      : {qc["passes_qc"]}')